In [ ]:
# ===========================================================================
# MODEL ANALYSIS - interactive driver (Phase 6 interpretability)
#
# Where notebooks/05_error_analysis.ipynb asks "what does the model get
# wrong", this notebook asks "what has the model actually learned": how the
# six ablation variants compare, how the learned embedding space is
# organised, what the attention-fusion model attends to, and which acoustic
# measurements drive a prediction.
#
#   src/model_analysis.py   ablation bar chart, embedding projections
#                            (t-SNE/PCA/UMAP), attention heatmaps, SHAP
#
# PREREQUISITES
#   1. notebooks/03_training.ipynb has produced at least one trained run,
#      i.e. outputs/metrics/phase2_comparison.csv and
#      outputs/embeddings/<RUN_NAME>/*.npz exist.
#   2. outputs/praat_features.csv (notebooks/02_feature_analysis.ipynb Stage 3)
#      for the SHAP analysis in the last section.
# ===========================================================================

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src import config
from src.console import print_header, print_kv
from src.error_analysis import load_run_predictions
from src.training.data import load_manifest

config.ensure_directories()

# Which trained run to analyse for the embedding/attention sections below -
# change this to compare models.
RUN_NAME = "detection_fusion"
TASK = "detection"

df_m6 = load_manifest()
preds = load_run_predictions(RUN_NAME)

print_header(f"Model Analysis - {RUN_NAME}")
print_kv("Manifest", f"{len(df_m6)} utterances")
print_kv("Predictions", f"{len(preds)} rows")

In [ ]:
# STAGE 1 - Ablation studies. The Phase 2/3 comparison table
# notebooks/03_training.ipynb produced, as a grouped bar chart - the visual
# counterpart to phase2_comparison.csv so the six-variant story doesn't rest
# on a reader scanning decimals.
from src.model_analysis import plot_ablation_comparison

comparison_df = pd.read_csv(config.METRICS_DIR / "phase2_comparison.csv", index_col=0)
ablation_figure = plot_ablation_comparison(
    comparison_df, title="Ablation Comparison - Detection", show=True)

print_header("Ablation Studies")
print_kv("Figure", ablation_figure)
comparison_df

In [ ]:
# STAGE 2 - Embedding visualization: t-SNE. Local structure of RUN_NAME's
# learned test-fold embeddings, coloured by true class, with misclassified
# points marked - answers whether the errors are scattered (genuinely
# ambiguous utterances) or clustered (a coherent region of the space the
# model has mislabelled).
from src.model_analysis import plot_embedding_map

tsne_figure = plot_embedding_map(RUN_NAME, preds, task=TASK, method="tsne", show=True)
print_kv("t-SNE figure", tsne_figure)

In [ ]:
# STAGE 3 - Embedding visualization: PCA. Linear, deterministic projection of
# the same embeddings - a sanity check for the nonlinear layouts above/below:
# if PCA already separates the classes, the fused embedding is doing most of
# the work linearly; if only t-SNE/UMAP separate them, the separation is
# genuinely nonlinear.
pca_figure = plot_embedding_map(RUN_NAME, preds, task=TASK, method="pca", show=True)
print_kv("PCA figure", pca_figure)

In [ ]:
# STAGE 4 - Embedding visualization: UMAP. Nonlinear like t-SNE, but tends to
# preserve more of the global distance structure between clusters - worth
# comparing against Stage 2's t-SNE layout before reading too much into
# either one's exact cluster shapes.
umap_figure = plot_embedding_map(RUN_NAME, preds, task=TASK, method="umap", show=True)
print_kv("UMAP figure", umap_figure)

In [ ]:
# STAGE 5 - Attention visualization (if available). Only AttentionFusionModel
# (model="attention_fusion", Ablation Model E) exposes attention_weights() -
# for any other model this prints a note and returns None rather than
# raising, since not every architecture has attention to visualize.
from src.model_analysis import plot_attention_heatmap

ATTENTION_RUN_NAME = "detection_attention_fusion"

attention_figure = plot_attention_heatmap(
    ATTENTION_RUN_NAME, task=TASK, model_name="attention_fusion", show=True)
print_kv("Attention figure", attention_figure or "skipped")

In [ ]:
# STAGE 6 - SHAP analysis + feature importance. Which of Phase 4's
# clinically-named Praat features drive the label, via a RandomForest
# surrogate fit on that same feature set (not SHAP run through the wav2vec
# 2.0 forward pass - see src.model_analysis.compute_shap_values for why).
# The mean-|SHAP-value| ranking below IS the feature-importance analysis -
# no separate machinery needed.
import numpy as np

from src.model_analysis import compute_shap_values, plot_shap_summary

praat_features = pd.read_csv(config.PRAAT_FEATURES_PATH)
shap_values, X_sample, feature_columns, class_names, surrogate = compute_shap_values(
    praat_features, task=TASK)
shap_figure = plot_shap_summary(
    shap_values, feature_columns, title=f"SHAP Feature Importance - {TASK}", show=True)

mean_abs = np.abs(shap_values).mean(axis=0)
top5 = pd.Series(mean_abs, index=feature_columns).sort_values(ascending=False).head(5)

print_header("SHAP Analysis")
print_kv("Figure", shap_figure)
print_kv("Top 5 features", ", ".join(top5.index))